# Q47: Contrast the Sobel and Canny methods for Edge Detection in Computer Vision
**Topic:** Computer-vision | **Difficulty:** Hard | **Time:** 10 min
**Sheet:** Grind75ML

---

## Question
Contrast the Sobel and Canny methods for Edge Detection in Computer Vision.

## Detailed Answer

### Sobel Edge Detection
**Type:** Gradient-based, first-order derivative filter

**How it works:**
1. Apply two 3×3 convolution kernels to compute horizontal and vertical gradients:

$$G_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix} * I \quad G_y = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1 \end{bmatrix} * I$$

2. Compute gradient magnitude: $G = \sqrt{G_x^2 + G_y^2}$
3. Compute gradient direction: $\theta = \arctan(G_y / G_x)$

**Characteristics:** Simple, fast, gives magnitude AND direction, but produces thick edges with noise.

### Canny Edge Detection
**Type:** Multi-stage algorithm (optimal edge detector)

**How it works:**
1. **Gaussian Blur**: Smooth image to reduce noise
2. **Gradient Computation**: Use Sobel to find gradients (magnitude + direction)
3. **Non-Maximum Suppression (NMS)**: Thin edges to 1-pixel width by keeping only local maxima along gradient direction
4. **Double Thresholding**: Classify edges as strong (> high threshold) or weak (between low and high)
5. **Hysteresis Edge Tracking**: Weak edges connected to strong edges are kept; isolated weak edges are discarded

### Comparison:
| Feature | Sobel | Canny |
|---------|-------|-------|
| Complexity | Simple (1 step) | Multi-stage (5 steps) |
| Edge thickness | Thick (multiple pixels) | Thin (1 pixel) |
| Noise sensitivity | High | Low (Gaussian pre-smoothing) |
| Parameters | None (or kernel size) | σ, low threshold, high threshold |
| Output | Gradient magnitude image | Binary edge map |
| Speed | Faster | Slower |
| Edge continuity | Gaps possible | Connected edges (hysteresis) |
| Use case | Gradient computation, feature input | Clean edge detection, contour finding |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def sobel_edge_detection(image: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Apply Sobel edge detection from scratch.
    
    Returns:
        (gradient_magnitude, gradient_direction)
    """
    # Sobel kernels
    Kx = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)
    Ky = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=float)
    
    padded = np.pad(image, 1, mode='edge')
    Gx = np.zeros_like(image, dtype=float)
    Gy = np.zeros_like(image, dtype=float)
    
    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            region = padded[i:i+3, j:j+3]
            Gx[i, j] = np.sum(region * Kx)
            Gy[i, j] = np.sum(region * Ky)
    
    magnitude = np.sqrt(Gx**2 + Gy**2)
    direction = np.arctan2(Gy, Gx)
    return magnitude, direction


def gaussian_blur(image: np.ndarray, sigma: float = 1.0) -> np.ndarray:
    """Apply Gaussian blur."""
    size = int(6 * sigma + 1) | 1  # ensure odd
    ax = np.arange(-size // 2 + 1, size // 2 + 1)
    xx, yy = np.meshgrid(ax, ax)
    kernel = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    kernel = kernel / kernel.sum()
    
    pad = size // 2
    padded = np.pad(image, pad, mode='edge')
    result = np.zeros_like(image, dtype=float)
    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            result[i, j] = np.sum(padded[i:i+size, j:j+size] * kernel)
    return result


def non_max_suppression(magnitude: np.ndarray, direction: np.ndarray) -> np.ndarray:
    """Thin edges to 1-pixel width."""
    h, w = magnitude.shape
    result = np.zeros_like(magnitude)
    angle = direction * 180 / np.pi
    angle[angle < 0] += 180
    
    for i in range(1, h-1):
        for j in range(1, w-1):
            # Find neighbors along gradient direction
            a = angle[i, j]
            if (0 <= a < 22.5) or (157.5 <= a <= 180):
                n1, n2 = magnitude[i, j-1], magnitude[i, j+1]
            elif 22.5 <= a < 67.5:
                n1, n2 = magnitude[i+1, j-1], magnitude[i-1, j+1]
            elif 67.5 <= a < 112.5:
                n1, n2 = magnitude[i-1, j], magnitude[i+1, j]
            else:
                n1, n2 = magnitude[i-1, j-1], magnitude[i+1, j+1]
            
            if magnitude[i, j] >= n1 and magnitude[i, j] >= n2:
                result[i, j] = magnitude[i, j]
    return result


def canny_edge_detection(image: np.ndarray, sigma: float = 1.0,
                          low_thresh: float = 0.05, high_thresh: float = 0.15) -> np.ndarray:
    """Canny edge detection from scratch."""
    # Step 1: Gaussian blur
    blurred = gaussian_blur(image, sigma)
    
    # Step 2: Gradient (using Sobel)
    magnitude, direction = sobel_edge_detection(blurred)
    magnitude = magnitude / magnitude.max()  # normalize
    
    # Step 3: Non-max suppression
    thin_edges = non_max_suppression(magnitude, direction)
    
    # Step 4: Double thresholding
    strong = thin_edges >= high_thresh
    weak = (thin_edges >= low_thresh) & (thin_edges < high_thresh)
    
    # Step 5: Hysteresis — connect weak edges to strong edges
    result = np.zeros_like(image)
    result[strong] = 255
    
    h, w = image.shape
    for i in range(1, h-1):
        for j in range(1, w-1):
            if weak[i, j]:
                if np.any(strong[i-1:i+2, j-1:j+2]):
                    result[i, j] = 255
    
    return result

In [ ]:
# --- Compare on a test image ---
np.random.seed(42)

# Create a synthetic test image
image = np.zeros((100, 100), dtype=float)
image[20:80, 20:80] = 200
image[40:60, 40:60] = 100
# Add circles
for i in range(100):
    for j in range(100):
        if (i-50)**2 + (j-75)**2 < 15**2:
            image[i, j] = 180
# Add noise
image += np.random.randn(100, 100) * 10

# Apply both methods
sobel_mag, _ = sobel_edge_detection(image)
canny_result = canny_edge_detection(image, sigma=1.0, low_thresh=0.1, high_thresh=0.3)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image, cmap='gray')
axes[0].set_title('Original Image')
axes[1].imshow(sobel_mag, cmap='gray')
axes[1].set_title('Sobel (Gradient Magnitude)')
axes[2].imshow(canny_result, cmap='gray')
axes[2].set_title('Canny (Binary Edge Map)')

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

print("Sobel: Thick edges, noisy, shows gradient strength")
print("Canny: Thin edges, clean, binary output")

## Key Takeaways
- **Sobel** is a building block — it computes gradients used BY Canny and other algorithms
- **Canny** is the standard for clean edge detection — 5 stages make it robust
- The **hysteresis** step in Canny is what makes edges connected and clean
- In deep learning, learned edge detectors (HED, BDCN) have surpassed both for semantic edges

In [ ]:
st = set()